# 12 · SOH forecasting benchmark (protocol D18)

**Task.** Forecast the SOH at RPT *k*+*h* (*h* = 1, 2, 3) from information available at RPT *k*. Labels use the C/20 capacity (D17). The protocol, feature sets, splits, methods and tests were fixed in `docs/DECISIONS.md` D18 **before** any model was trained; deviations and disclosures are listed there.

- **P1, continual stream (headline).** Batteries arrive one by one (10 orders × 5 seeds). After every task, all tasks' test sets are evaluated.
- **P2, unseen battery.** Leave one battery out.
- **Feature sets.**
  - F0: SOH history, static first-life and duty descriptors, and the known-future schedule.
  - F1: F0 plus the 47 diagnostic features (notebook 11).
- **Reference forecasts.** Persistence (no change) and linear extrapolation are always shown.

## 1. Samples

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

import torch
torch.set_num_threads(1)
from src import experiments as X
from src import evaluation as ev
from src import forecasting as FC

dp.check_data_files(DATA_ROOT, required=["diagnostic_features.csv", "cleaned_battery_data.csv"])
table = dp.read_dataset(DATA_ROOT / "diagnostic_features.csv")
temp_col = next(c for c in dp.read_header(DATA_ROOT / "cleaned_battery_data.csv") if c.startswith("Aux_Temperature"))
points = dp.read_dataset(DATA_ROOT / "cleaned_battery_data.csv", usecols=["Date_Time", "Source_Folder", "Cycling_Test", temp_col])
blocks = FC.cycling_blocks(points)
samples = FC.build_samples(table, blocks)
SETS = FC.feature_sets(samples)
split = FC.assign_split(samples)
print(f"{len(samples)} samples; feature sets: " + ", ".join(f"{k}={len(v)}" for k, v in SETS.items()))
display(pd.crosstab([split[dp.GROUP_COL]], [split["split"], split["horizon"]]))
display(blocks.pivot(index="Cycling_Test", columns=dp.GROUP_COL, values="setpoint_c")
        .loc[sorted(set(blocks["Cycling_Test"]), key=lambda s: int(s.split("_")[1]))].T)

## 2. Hyperparameter tuning (validation only, D11)

Stage 1 selects each backbone's learning rate by validation plasticity. Stage 2 selects the method hyperparameters (EWC λ, replay buffer) by final average validation error. Tuning uses three orders disjoint from the ten evaluation orders.

In [ ]:
base_cfg = FC.cl_config(SETS["F1"])
tuning = X.tune_hyperparameters(samples, base_cfg, X.TuningConfig(), stream_builder=FC.forecast_stream)
CFG = tuning["tuned_cl_config"]
print("backbone:", tuning["backbone_selected"])
print("methods:", tuning["selected"])
print("at grid edge:", tuning["at_grid_edge"])
display(tuning["grid_summary"].round(4))

## 3. P1: continual stream

In [ ]:
p1 = FC.run_p1(samples, CFG)
cfg_f0 = FC.cl_config(SETS["F0"], methods=("tft_ewc",))
cfg_f0.backbone_kwargs, cfg_f0.method_kwargs = CFG.backbone_kwargs, CFG.method_kwargs     # same tuned settings
p1_f0 = FC.run_p1(samples, cfg_f0, label="_F0")
runs = pd.concat([p1["runs"], p1_f0["runs"]], ignore_index=True)
final = pd.concat([p1["final_per_task"], p1_f0["final_per_task"]], ignore_index=True)
joint = FC.run_joint(samples, CFG)
base_p1 = FC.run_baselines_p1(samples, SETS)

cl_summary = ev.summarise_runs(runs, ["final_avg_error", "plasticity", "backward_transfer", "forgetting", "forward_transfer"], ["method"])
display(cl_summary.round(3))

In [ ]:
# Final-step test RMSE per battery, averaged over tasks: CL methods (mean over order x seed), joint (mean over seeds), baselines.
def per_method(frame, keys=("method",)):
    per_run = frame.groupby([*keys, *[c for c in ("order_id", "seed") if c in frame]], dropna=False)["RMSE"].mean()
    return per_run.groupby(list(keys)).agg(["mean", "std", "size"])

rmse_table = pd.concat([per_method(final), per_method(joint), per_method(base_p1.assign(seed=0))]).sort_values("mean")
persist = rmse_table.loc["persistence", "mean"]
rmse_table["skill_vs_persistence"] = 1 - rmse_table["mean"] / persist
display(rmse_table.round(3))

h_cols = [c for c in final.columns if c.startswith("RMSE_h")]
by_h = pd.concat([final.groupby("method")[h_cols].mean(), joint.groupby("method")[h_cols].mean(),
                  base_p1.groupby("method")[h_cols].mean()])
display(by_h.loc[rmse_table.index].round(3))

### Paired tests (fixed family, D18)

- **Methods vs TFT + EWC:** each continual method against TFT + EWC on matched (order, seed) runs, per metric, with Holm correction.
- **Diagnostics:** F1 vs F0 for TFT + EWC.
- **Baselines:** TFT + EWC's final average error against each baseline's (a constant over runs), with a one-sample Wilcoxon test on the differences.

In [ ]:
comp = X.compare_methods(runs, baseline="tft_ewc", metric_cols=["final_avg_error", "forgetting", "backward_transfer"])
display(comp[["metric", "method", "mean_diff", "ci_low", "ci_high", "p_value", "p_holm", "rank_biserial"]].round(4)
        if "mean_diff" in comp else comp.round(4))

tft = runs[runs["method"] == "tft_ewc"].sort_values(["order_id", "seed"])["final_avg_error"].to_numpy()
base_final = base_p1.groupby("method")["RMSE"].mean()
vs_base, pv = [], {}
for name, value in base_final.items():
    res = ev.paired_comparison(tft, np.full(len(tft), value))
    vs_base.append({"baseline": name, "baseline_final_avg_error": value, **res})
    pv[name] = res["p_value"]
holm = ev.holm_correction(pv)
vs_base = pd.DataFrame(vs_base).assign(p_holm=lambda d: d["baseline"].map(holm))
display(vs_base.round(4))

In [ ]:
order = [m for m in rmse_table.index]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
colors = ["#2a78d6" if m.startswith("tft") else "#8a8981" if m in base_final.index else "#1baf7a" for m in order]
ax.barh(order, rmse_table["mean"], xerr=rmse_table["std"].fillna(0), color=colors, height=0.6, error_kw={"elinewidth": 1})
for i, v in enumerate(rmse_table["mean"]):
    ax.text(v, i, f"  {v:.2f}", va="center", fontsize=8, color="#3d3d3a")
ax.axvline(persist, color="#1a1a19", linewidth=1, linestyle="--")
ax.invert_yaxis()
ax.set_xlabel("Test RMSE of SOH(k+h), pp (mean over batteries; ±1 sd over runs)")
ax.set_title("P1: final-step forecast error (blue: TFT, green: MLP, grey: baselines)", loc="left")
ax.grid(axis="y", visible=False)
plt.tight_layout()
savefig(fig, "12_p1_rmse")
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.8))
show = ["persistence", "linear_extrapolation", "ridge_schedule", "gbm_F1", "finetune", "tft_ewc", "tft_ewc_F0", "tft_joint"]
markers = dict(zip(show, ["o", "s", "^", "D", "v", "P", "X", "*"]))
for m in show:
    ax.plot([1, 2, 3], by_h.loc[m, h_cols].to_numpy(), marker=markers[m], label=m,
            color="#2a78d6" if m.startswith("tft") else "#1baf7a" if m == "finetune" else "#8a8981",
            linestyle="-" if m.startswith("tft") else "--")
ax.set_xticks([1, 2, 3]); ax.set_xlabel("Horizon h (RPTs ahead)"); ax.set_ylabel("Test RMSE (SOH pp)")
ax.set_title("P1: error by horizon", loc="left")
ax.legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout()
savefig(fig, "12_p1_horizon")
plt.show()

In [ ]:
# Mean accuracy matrix of TFT + EWC in stream position coordinates (row: after task i; column: task j).
mats = [m.to_numpy(dtype=float) for (name, _, _), m in p1["matrices"].items() if name == "tft_ewc"]
M = np.nanmean(np.stack(mats), axis=0)
fig, ax = plt.subplots(figsize=(4.6, 3.9))
im = ax.imshow(M, cmap="Blues", vmin=0)
for (i, j), v in np.ndenumerate(M):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > np.nanmax(M) * 0.6 else "#1a1a19")
ax.set_xlabel("Task position j"); ax.set_ylabel("After training task i")
ax.set_xticks(range(M.shape[1]), range(1, M.shape[1] + 1)); ax.set_yticks(range(M.shape[0]), range(1, M.shape[0] + 1))
ax.set_title("TFT + EWC: test RMSE (pp), mean of 50 runs", loc="left", fontsize=10)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8)
savefig(fig, "12_tft_ewc_matrix")
plt.show()

## 4. P2: unseen battery (leave one battery out)

In [ ]:
p2 = FC.run_p2(samples, CFG, SETS)
p2_battery = p2.groupby(["method", dp.GROUP_COL])["RMSE"].mean().unstack()
p2_table = p2_battery.assign(mean=p2_battery.mean(axis=1)).sort_values("mean")
display(p2_table.round(3))
pv2, rows2 = {}, []
for m in p2_table.index.drop("tft"):
    res = ev.paired_comparison(p2_battery.loc["tft"].to_numpy(), p2_battery.loc[m].to_numpy())
    rows2.append({"method": m, **res}); pv2[m] = res["p_value"]
h2 = ev.holm_correction(pv2)
p2_tests = pd.DataFrame(rows2).assign(p_holm=lambda d: d["method"].map(h2))
display(p2_tests.round(4))

## 5. Example forecasts

In [ ]:
from src.continual_learning import TFTJoint
from src.experiments import method_settings
held = "v5"
task = FC.leave_one_out_task(samples, held)
kw = method_settings(CFG, "tft_joint"); feats = list(kw.pop("features"))
tft_m = TFTJoint(feats, seed=0, history=samples, **kw).fit_all([task])
test = task.test.assign(pred_tft=tft_m.predict(task.test))
fit = pd.concat([task.train, task.val])
bp = FC.baseline_predictions(fit, {"t": test}, SETS)
test["pred_lin"] = bp["linear_extrapolation"]["t"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, h in zip(axes, (1, 2, 3)):
    d = test[test["horizon"] == h].sort_values("target_date")
    ax.plot(d["target_date"], d["soh_target"], "o-", color="#1a1a19", label="measured", markersize=4)
    ax.plot(d["target_date"], d["soh_origin"] + d["pred_tft"], "P--", color="#2a78d6", label="TFT (P2)", markersize=5)
    ax.plot(d["target_date"], d["soh_origin"] + d["pred_lin"], "s:", color="#8a8981", label="linear extrap.", markersize=4)
    ax.set_title(f"{held}, h = {h}", loc="left"); ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("SOH at target RPT (%)"); axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
savefig(fig, "12_example_forecasts")
plt.show()

## 6. Save results

In [ ]:
out = X.write_results(RESULTS_DIR / "forecasting.json", {
    "protocol": "docs/DECISIONS.md D18", "label_basis": "C/20 capacity (D17)",
    "n_samples": int(len(samples)), "feature_sets": {k: list(v) for k, v in SETS.items()},
    "tuning": {"backbone_selected": tuning["backbone_selected"], "selected": tuning["selected"],
               "at_grid_edge": tuning["at_grid_edge"]},
    "p1_cl_summary": cl_summary, "p1_rmse": rmse_table.reset_index(names="method"),
    "p1_rmse_by_horizon": by_h.reset_index(names="method"), "p1_tests_vs_tft_ewc": comp,
    "p1_tft_ewc_vs_baselines": vs_base, "p2_rmse": p2_table.reset_index(names="method"), "p2_tests_vs_tft": p2_tests,
}, REPO_ROOT)
print("wrote", out.relative_to(REPO_ROOT))

## 7. Interpretation

Written from the executed run in `docs/DATA_REPORT.md` §7; the numbers are in `results/forecasting.json`.